# Air-training analysis: full air-cycle organization

The goal of this analysis is to determine how air delivery structures locomotor behavior across the full air-on and air-off cycle.

Rather than using post-pre delta values as the dependent variable, we model actual locomotor values from sequential windows. This allows us to compare biologically meaningful epochs directly while preserving absolute locomotor levels.

The main air-cycle sequence is:

pre_air_on → post_air_on → pre_air_on_mid → post_air_on_mid → pre_air_off → post_air_off → pre_air_off_mid → post_air_off_mid → pre_air_on_next

Planned comparisons:

1. Air-onset transition:
   pre_air_on vs post_air_on

2. Early air-on progression:
   post_air_on vs pre_air_on_mid

3. Late air-on progression:
   post_air_on_mid vs pre_air_off

4. Air-offset transition:
   pre_air_off vs post_air_off

5. Early air-off recovery:
   post_air_off vs pre_air_off_mid

6. Late air-off recovery / return toward next baseline:
   post_air_off_mid vs pre_air_on_next

Outcomes are analyzed in the following order:
1. fraction moving
2. fraction forward
3. path speed and net speed
4. path distance and net distance

For each outcome and planned comparison, linear mixed-effects models are used. Basic models test whether the second epoch differs from the first. Time-dependent models test whether this difference changes across air-training sessions and within-session time.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

encoder_epoch_df = pd.read_hdf(
    Path(pdata_root) / "_cache" / "behavior_epoch_metrics.h5",
    key="encoder/prepost_1s_speedThresh_1cms"
)

from src.qc.qc_events import load_behavior_qc_tables

events_df, session_summary_df = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

from src.utils.pdata_organize import make_session_availability_summary

session_availability_df = make_session_availability_summary(
    events_df=events_df,
    windows_df=windows_df,
    min_session_duration_s=900,
    min_valid_events=3,
)

from src.utils.pdata_organize import add_day_bins_to_sessions

session_day_df = add_day_bins_to_sessions(
    session_availability_df,
    use_good_sessions_only=True,
)

Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (103980, 40)
Valid windows: (101861, 40)
[LOADED] Behavior QC tables: /mnt/pdata/Classical_Conditioning/_cache/behavior_QC.h5


In [4]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

air_phase = "air_training"

air_anchors = [
    "air_on",
    "air_on_mid",
    "air_off",
    "air_off_mid",
]

# Candidate outcome names. Keep only those present in encoder_epoch_df.
candidate_outcomes = [
    "frac_moving",
    "frac_forward",
    "frac_stationary",
    "mean_speed_path_cms",
    "mean_speed_net_cms",
    "distance_path_cm",
    "distance_net_cm",
    "dist_path_cm",
    "dist_net_cm",
]

air_outcomes = [
    c for c in candidate_outcomes
    if c in encoder_epoch_df.columns
]

print("Available air outcomes:")
print(air_outcomes)

print("\nDistance-like columns:")
print([c for c in encoder_epoch_df.columns if "dist" in c.lower() or "distance" in c.lower()])

Available air outcomes:
['frac_moving', 'frac_forward', 'frac_stationary', 'mean_speed_path_cms', 'mean_speed_net_cms', 'distance_path_cm', 'distance_net_cm']

Distance-like columns:
['distance_path_cm', 'distance_net_cm']


In [5]:
def prepare_air_window_df(
    encoder_epoch_df,
    phase="air_training",
    anchors=None,
    keep_overlap=True,
    require_good_session=True,
    require_valid_window=True,
):
    """
    Prepare long-format air-training window dataframe.

    One row = one pre/post window around one air-related anchor.
    """

    if anchors is None:
        anchors = [
            "air_on",
            "air_on_mid",
            "air_off",
            "air_off_mid",
        ]

    df = encoder_epoch_df.copy()

    df = df[
        (df["phase"] == phase) &
        (df["anchor_name"].isin(anchors)) &
        (df["window_position"].isin(["pre", "post"]))
    ].copy()

    if require_good_session and "good_session_basic" in df.columns:
        df = df[df["good_session_basic"] == True].copy()

    if require_valid_window and "valid_window" in df.columns:
        df = df[df["valid_window"] == True].copy()

    if not keep_overlap and "overlap_flag" in df.columns:
        df = df[df["overlap_flag"] == False].copy()

    # --------------------------------------------------
    # Create biological epoch labels
    # --------------------------------------------------
    label_map = {
        ("air_on", "pre"): "pre_air_on",
        ("air_on", "post"): "post_air_on",

        ("air_on_mid", "pre"): "pre_air_on_mid",
        ("air_on_mid", "post"): "post_air_on_mid",

        ("air_off", "pre"): "pre_air_off",
        ("air_off", "post"): "post_air_off",

        ("air_off_mid", "pre"): "pre_air_off_mid",
        ("air_off_mid", "post"): "post_air_off_mid",
    }

    df["epoch_label"] = [
        label_map.get((a, w), np.nan)
        for a, w in zip(df["anchor_name"], df["window_position"])
    ]

    df = df[df["epoch_label"].notna()].copy()

    # --------------------------------------------------
    # Random-effect IDs
    # --------------------------------------------------
    df["animal_day"] = (
        df["animal"].astype(str) + ":" +
        df["date"].astype(str)
    )

    # --------------------------------------------------
    # Session time
    # --------------------------------------------------
    if "session_time_min" not in df.columns:
        if "anchor_session_time_min" in df.columns:
            df["session_time_min"] = df["anchor_session_time_min"]
        elif "anchor_time_s" in df.columns:
            df["session_time_min"] = df["anchor_time_s"] / 60.0
        elif "session_time_s" in df.columns:
            df["session_time_min"] = df["session_time_s"] / 60.0
        else:
            df["session_time_min"] = np.nan

    # --------------------------------------------------
    # Exposure session
    # --------------------------------------------------
    if "phase_session_number" in df.columns:
        df["exposure_session"] = df["phase_session_number"]
    elif "phase_day_number_good" in df.columns:
        df["exposure_session"] = df["phase_day_number_good"]
    else:
        df["exposure_session"] = np.nan

    return df.reset_index(drop=True)

In [6]:
air_window_df = prepare_air_window_df(
    encoder_epoch_df,
    phase=air_phase,
    anchors=air_anchors,
    keep_overlap=True,
    require_good_session=True,
    require_valid_window=True,
)

print(air_window_df.shape)
print(air_window_df["epoch_label"].value_counts())

(28229, 63)
epoch_label
pre_air_off         3535
post_air_on_mid     3535
post_air_on         3535
pre_air_on_mid      3534
pre_air_on          3532
post_air_off        3525
pre_air_off_mid     3517
post_air_off_mid    3516
Name: count, dtype: int64


In [7]:
air_window_df[
    [
        "animal",
        "date",
        "phase",
        "anchor_name",
        "window_position",
        "epoch_label",
        "event_number",
        "session_time_min",
        "exposure_session",
    ]
].head(20)

,animal,date,phase,anchor_name,window_position,epoch_label,event_number,session_time_min,exposure_session
0,NML_04,2026_01_12,air_training,air_off_mid,post,post_air_off_mid,0,0.206890,1
1,NML_04,2026_01_12,air_training,air_off_mid,pre,pre_air_off_mid,0,0.206890,1
2,NML_04,2026_01_12,air_training,air_off,post,post_air_off,0,0.081173,1
3,NML_04,2026_01_12,air_training,air_off,pre,pre_air_off,0,0.081173,1
4,NML_04,2026_01_12,air_training,air_on_mid,post,post_air_on_mid,0,0.040587,1
5,NML_04,2026_01_12,air_training,air_on_mid,pre,pre_air_on_mid,0,0.040587,1
6,NML_04,2026_01_12,air_training,air_on,post,post_air_on,0,0.000000,1
7,NML_04,2026_01_12,air_training,air_off_mid,post,post_air_off_mid,1,0.525370,1
8,NML_04,2026_01_12,air_training,air_off_mid,pre,pre_air_off_mid,1,0.525370,1
9,NML_04,2026_01_12,air_training,air_off,post,post_air_off,1,0.399650,1


In [8]:
air_comparisons = [
    {
        "comparison": "air_onset_transition",
        "epoch1": "pre_air_on",
        "epoch2": "post_air_on",
        "epoch2_trial_shift": 0,
        "interpretation": "Air-onset transition",
    },
    {
        "comparison": "early_air_on_progression",
        "epoch1": "post_air_on",
        "epoch2": "pre_air_on_mid",
        "epoch2_trial_shift": 0,
        "interpretation": "Early air-on progression",
    },
    {
        "comparison": "air_on_mid_transition",
        "epoch1": "pre_air_on_mid",
        "epoch2": "post_air_on_mid",
        "epoch2_trial_shift": 0,
        "interpretation": "Air-on midpoint transition",
    },
    {
        "comparison": "late_air_on_progression",
        "epoch1": "post_air_on_mid",
        "epoch2": "pre_air_off",
        "epoch2_trial_shift": 0,
        "interpretation": "Late air-on progression",
    },
    {
        "comparison": "air_offset_transition",
        "epoch1": "pre_air_off",
        "epoch2": "post_air_off",
        "epoch2_trial_shift": 0,
        "interpretation": "Air-offset transition",
    },
    {
        "comparison": "early_air_off_recovery",
        "epoch1": "post_air_off",
        "epoch2": "pre_air_off_mid",
        "epoch2_trial_shift": 0,
        "interpretation": "Early air-off recovery",
    },
    {
        "comparison": "air_off_mid_transition",
        "epoch1": "pre_air_off_mid",
        "epoch2": "post_air_off_mid",
        "epoch2_trial_shift": 0,
        "interpretation": "Air-off midpoint transition",
    },
    {
        "comparison": "late_air_off_recovery_to_next_baseline",
        "epoch1": "post_air_off_mid",
        "epoch2": "pre_air_on",
        "epoch2_trial_shift": 1,
        "interpretation": "Late air-off recovery / next pre-air baseline",
    },
]

pd.DataFrame(air_comparisons)

,comparison,epoch1,epoch2,epoch2_trial_shift,interpretation
0,air_onset_transition,pre_air_on,post_air_on,0,Air-onset transition
1,early_air_on_progression,post_air_on,pre_air_on_mid,0,Early air-on progression
2,air_on_mid_transition,pre_air_on_mid,post_air_on_mid,0,Air-on midpoint transition
3,late_air_on_progression,post_air_on_mid,pre_air_off,0,Late air-on progression
4,air_offset_transition,pre_air_off,post_air_off,0,Air-offset transition
5,early_air_off_recovery,post_air_off,pre_air_off_mid,0,Early air-off recovery
6,air_off_mid_transition,pre_air_off_mid,post_air_off_mid,0,Air-off midpoint transition
7,late_air_off_recovery_to_next_baseline,post_air_off_mid,pre_air_on,1,Late air-off recovery / next pre-air baseline


In [9]:
def make_air_pair_df(
    air_window_df,
    comparison,
    epoch1,
    epoch2,
    epoch2_trial_shift=0,
    trial_col="event_number",
):
    """
    Build long-format paired dataframe for one planned comparison.

    epoch1 is the first window.
    epoch2 is the second window.

    If epoch2_trial_shift = 1, then epoch2 from trial N+1 is paired
    with epoch1 from trial N. This is used for:

        post_air_off_mid trial N vs pre_air_on trial N+1
    """

    df = air_window_df.copy()

    required_cols = [
        "animal",
        "date",
        trial_col,
        "epoch_label",
        "session_time_min",
        "exposure_session",
        "animal_day",
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    e1 = df[df["epoch_label"] == epoch1].copy()
    e2 = df[df["epoch_label"] == epoch2].copy()

    # Pairing trial number
    e1["pair_trial"] = e1[trial_col].astype(int)
    e2["pair_trial"] = e2[trial_col].astype(int) - int(epoch2_trial_shift)

    pair_keys = ["animal", "date", "pair_trial"]

    # Keep only pairs present in both epochs
    common_pairs = (
        e1[pair_keys]
        .drop_duplicates()
        .merge(
            e2[pair_keys].drop_duplicates(),
            on=pair_keys,
            how="inner"
        )
    )

    e1 = e1.merge(common_pairs, on=pair_keys, how="inner")
    e2 = e2.merge(common_pairs, on=pair_keys, how="inner")

    # Pair-level time and exposure are taken from the first epoch
    pair_info = (
        e1[pair_keys + ["session_time_min", "exposure_session"]]
        .drop_duplicates(subset=pair_keys)
        .rename(
            columns={
                "session_time_min": "pair_session_time_min",
                "exposure_session": "pair_exposure_session",
            }
        )
    )

    e1 = e1.merge(pair_info, on=pair_keys, how="left")
    e2 = e2.merge(pair_info, on=pair_keys, how="left")

    e1["comparison"] = comparison
    e2["comparison"] = comparison

    e1["comparison_epoch"] = epoch1
    e2["comparison_epoch"] = epoch2

    e1["epoch2_indicator"] = 0
    e2["epoch2_indicator"] = 1

    pair_df = pd.concat([e1, e2], ignore_index=True)

    pair_df["pair_id"] = (
        pair_df["animal"].astype(str) + ":" +
        pair_df["date"].astype(str) + ":" +
        pair_df["comparison"].astype(str) + ":" +
        pair_df["pair_trial"].astype(str)
    )

    return pair_df.reset_index(drop=True)

In [10]:
air_pair_list = []

for comp in air_comparisons:
    tmp = make_air_pair_df(
        air_window_df,
        comparison=comp["comparison"],
        epoch1=comp["epoch1"],
        epoch2=comp["epoch2"],
        epoch2_trial_shift=comp["epoch2_trial_shift"],
    )
    air_pair_list.append(tmp)

air_pair_df = pd.concat(air_pair_list, ignore_index=True)

print(air_pair_df.shape)
print(air_pair_df["comparison"].value_counts())
print(air_pair_df.groupby(["comparison", "comparison_epoch"])["pair_id"].nunique())

(56292, 70)
comparison
late_air_on_progression                   7068
early_air_on_progression                  7066
air_on_mid_transition                     7066
air_onset_transition                      7062
air_offset_transition                     7048
early_air_off_recovery                    7034
air_off_mid_transition                    7032
late_air_off_recovery_to_next_baseline    6916
Name: count, dtype: int64
comparison                              comparison_epoch
air_off_mid_transition                  post_air_off_mid    3516
                                        pre_air_off_mid     3516
air_offset_transition                   post_air_off        3524
                                        pre_air_off         3524
air_on_mid_transition                   post_air_on_mid     3533
                                        pre_air_on_mid      3533
air_onset_transition                    post_air_on         3531
                                        pre_air_on          353

In [11]:
air_pair_df["pair_exposure_session_c"] = (
    air_pair_df["pair_exposure_session"] -
    air_pair_df["pair_exposure_session"].mean()
)

air_pair_df["pair_session_10m_c"] = (
    air_pair_df["pair_session_time_min"] -
    air_pair_df["pair_session_time_min"].mean()
) / 10.0

air_pair_df[
    [
        "animal",
        "date",
        "comparison",
        "comparison_epoch",
        "epoch2_indicator",
        "pair_trial",
        "pair_exposure_session",
        "pair_exposure_session_c",
        "pair_session_time_min",
        "pair_session_10m_c",
    ]
].head(20)

,animal,date,comparison,comparison_epoch,epoch2_indicator,pair_trial,pair_exposure_session,pair_exposure_session_c,pair_session_time_min,pair_session_10m_c
0,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,1,1,-7.11618,0.332607,-1.012639
1,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,2,1,-7.11618,0.651090,-0.980791
2,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,3,1,-7.11618,0.939413,-0.951959
3,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,4,1,-7.11618,1.221443,-0.923756
4,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,5,1,-7.11618,1.508487,-0.895051
5,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,6,1,-7.11618,1.947493,-0.851151
6,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,7,1,-7.11618,2.219347,-0.823965
7,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,8,1,-7.11618,2.502110,-0.795689
8,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,9,1,-7.11618,2.770740,-0.768826
9,NML_04,2026_01_12,air_onset_transition,pre_air_on,0,10,1,-7.11618,3.063853,-0.739515


In [20]:
def fit_air_basic_pair_model(
    air_pair_df,
    outcome,
    comparison,
):
    """
    Basic planned comparison using actual values.

    outcome ~ epoch2_indicator

    epoch2_indicator coefficient:
        value in second epoch - value in first epoch
    """

    df = air_pair_df.copy()
    df = df[df["comparison"] == comparison].copy()
    df = df[np.isfinite(df[outcome])].copy()

    model = smf.mixedlm(
        f"{outcome} ~ epoch2_indicator",
        data=df,
        groups=df["animal"],
        re_formula="1",
        vc_formula={
            "animal_day": "0 + C(animal_day)",
            "pair_id": "0 + C(pair_id)",
        }
    ).fit(
        reml=False,
        method="lbfgs",
        maxiter=1000,
    )

    return model

In [30]:
from joblib import Parallel, delayed, parallel_backend
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [31]:
def run_one_air_basic_model(outcome, comp, air_pair_df):
    """
    Fit one basic air-cycle planned-comparison model.

    Returns:
        key = (outcome, comparison)
        model = fitted statsmodels result or None
        status info
    """

    comparison = comp["comparison"]
    interpretation = comp["interpretation"]

    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")

            model = fit_air_basic_pair_model(
                air_pair_df,
                outcome=outcome,
                comparison=comparison,
            )

            warning_text = " | ".join(
                sorted(set(str(x.message) for x in w))
            )

        status = {
            "outcome": outcome,
            "comparison": comparison,
            "interpretation": interpretation,
            "success": True,
            "converged": getattr(model, "converged", np.nan),
            "n_obs": int(model.nobs),
            "warnings": warning_text,
            "error": "",
        }

        return (outcome, comparison), model, status

    except Exception as e:
        status = {
            "outcome": outcome,
            "comparison": comparison,
            "interpretation": interpretation,
            "success": False,
            "converged": False,
            "n_obs": np.nan,
            "warnings": "",
            "error": str(e),
        }

        return (outcome, comparison), None, status

In [32]:
jobs = []

for outcome in air_outcomes:
    for comp in air_comparisons:
        jobs.append((outcome, comp))

len(jobs)

56

In [33]:
n_jobs = 4   # start with 4; increase only if memory is okay

with parallel_backend("loky", inner_max_num_threads=1):
    parallel_results = Parallel(
        n_jobs=n_jobs,
        verbose=10
    )(
        delayed(run_one_air_basic_model)(
            outcome,
            comp,
            air_pair_df
        )
        for outcome, comp in jobs
    )

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:  3.0min
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  6.5min
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed: 10.0min
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed: 13.7min
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed: 20.4min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed: 25.0min
[Parallel(n_jobs=4)]: Done  56 out of  56 | elapsed: 33.9min finished


In [34]:
air_basic_models = {}
status_rows = []

for key, model, status in parallel_results:
    status_rows.append(status)

    if model is not None:
        air_basic_models[key] = model

air_basic_status_df = pd.DataFrame(status_rows)

air_basic_status_df

,outcome,comparison,interpretation,success,converged,n_obs,warnings,error
0,frac_moving,air_onset_transition,Air-onset transition,True,True,7062,The Hessian matrix at the estimated parameter ...,
1,frac_moving,early_air_on_progression,Early air-on progression,True,True,7066,The Hessian matrix at the estimated parameter ...,
2,frac_moving,air_on_mid_transition,Air-on midpoint transition,True,True,7066,,
3,frac_moving,late_air_on_progression,Late air-on progression,True,True,7068,The Hessian matrix at the estimated parameter ...,
4,frac_moving,air_offset_transition,Air-offset transition,True,False,7048,"Gradient optimization failed, |grad| = 75.1800...",
5,frac_moving,early_air_off_recovery,Early air-off recovery,True,True,7034,The Hessian matrix at the estimated parameter ...,
6,frac_moving,air_off_mid_transition,Air-off midpoint transition,True,True,7032,The MLE may be on the boundary of the paramete...,
7,frac_moving,late_air_off_recovery_to_next_baseline,Late air-off recovery / next pre-air baseline,True,False,6916,"Gradient optimization failed, |grad| = 17.1077...",
8,frac_forward,air_onset_transition,Air-onset transition,True,False,7062,"Gradient optimization failed, |grad| = 73.9367...",
9,frac_forward,early_air_on_progression,Early air-on progression,True,True,7066,The MLE may be on the boundary of the paramete...,


In [35]:
air_basic_status_df[
    (air_basic_status_df["success"] == False) |
    (air_basic_status_df["converged"] == False)
]

,outcome,comparison,interpretation,success,converged,n_obs,warnings,error
4,frac_moving,air_offset_transition,Air-offset transition,True,False,7048,"Gradient optimization failed, |grad| = 75.1800...",
7,frac_moving,late_air_off_recovery_to_next_baseline,Late air-off recovery / next pre-air baseline,True,False,6916,"Gradient optimization failed, |grad| = 17.1077...",
8,frac_forward,air_onset_transition,Air-onset transition,True,False,7062,"Gradient optimization failed, |grad| = 73.9367...",
13,frac_forward,early_air_off_recovery,Early air-off recovery,True,False,7034,"Gradient optimization failed, |grad| = 38.8941...",
23,frac_stationary,late_air_off_recovery_to_next_baseline,Late air-off recovery / next pre-air baseline,True,False,6916,"Gradient optimization failed, |grad| = 17.1077...",
26,mean_speed_path_cms,air_on_mid_transition,Air-on midpoint transition,True,False,7066,"Gradient optimization failed, |grad| = 141.583...",
33,mean_speed_net_cms,early_air_on_progression,Early air-on progression,True,False,7066,"Gradient optimization failed, |grad| = 141.348...",
41,distance_path_cm,early_air_on_progression,Early air-on progression,True,False,7066,"Gradient optimization failed, |grad| = 141.881...",
49,distance_net_cm,early_air_on_progression,Early air-on progression,True,False,7066,"Gradient optimization failed, |grad| = 141.346...",
50,distance_net_cm,air_on_mid_transition,Air-on midpoint transition,True,False,7066,"Gradient optimization failed, |grad| = 137.762...",


In [36]:
basic_rows = []

for outcome in air_outcomes:
    for comp in air_comparisons:
        key = (outcome, comp["comparison"])

        if key not in air_basic_models:
            continue

        basic_rows.append(
            extract_air_basic_result(
                air_basic_models[key],
                outcome=outcome,
                comparison=comp["comparison"],
                interpretation=comp["interpretation"],
            )
        )

air_basic_results_df = pd.DataFrame(basic_rows)

air_basic_results_df["outcome_order"] = air_basic_results_df["outcome"].map(outcome_order)
air_basic_results_df["comparison_order"] = air_basic_results_df["comparison"].map(comparison_order)

air_basic_results_df = (
    air_basic_results_df
    .sort_values(["outcome_order", "comparison_order"])
    .reset_index(drop=True)
)

air_basic_results_df

NameError: name 'extract_air_basic_result' is not defined

In [ ]:
air_basic_models = {}

for outcome in air_outcomes:
    for comp in air_comparisons:
        comparison = comp["comparison"]

        try:
            model = fit_air_basic_pair_model(
                air_pair_df,
                outcome=outcome,
                comparison=comparison,
            )

            air_basic_models[(outcome, comparison)] = model

            print("\n\n==============================")
            print(outcome, comparison)
            print(comp["interpretation"])
            print("==============================")
            print(model.summary())

        except Exception as e:
            print(f"[ERROR] {outcome}, {comparison}: {e}")

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)




frac_moving air_onset_transition
Air-onset transition
          Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: frac_moving
No. Observations:  7062    Method:             ML         
No. Groups:        5       Scale:              0.0867     
Min. group size:   880     Log-Likelihood:     -1494.9130 
Max. group size:   1690    Converged:          Yes        
Mean group size:   1412.4                                 
----------------------------------------------------------
                 Coef. Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept        0.290    0.126  2.307 0.021  0.044  0.536
epoch2_indicator 0.501    0.007 71.449 0.000  0.487  0.514
Group Var        0.078                                    
animal_day Var   0.011    0.007                           
pair_id Var      0.000    0.004                           



/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2207: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2219: ConvergenceWarning: Gradient optimization failed, |grad| = 28.841050
  warnings.warn(msg, ConvergenceWarning)
/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/nmldata2/miniconda3/envs/air



frac_moving early_air_on_progression
Early air-on progression
          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: frac_moving
No. Observations: 7066    Method:             ML         
No. Groups:       5       Scale:              0.0514     
Min. group size:  880     Log-Likelihood:     334.8360   
Max. group size:  1690    Converged:          No         
Mean group size:  1413.2                                 
---------------------------------------------------------
                 Coef. Std.Err.   z   P>|z| [0.025 0.975]
---------------------------------------------------------
Intercept        0.772    0.079 9.719 0.000  0.617  0.928
epoch2_indicator 0.049    0.005 9.033 0.000  0.038  0.059
Group Var        0.031                                   
animal_day Var   0.008    0.006                          
pair_id Var      0.000    0.004                          



/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


...



distance_net_cm air_offset_transition
Air-offset transition
            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: distance_net_cm
No. Observations: 7048    Method:             ML             
No. Groups:       5       Scale:              53.9681        
Min. group size:  872     Log-Likelihood:     -24183.2492    
Max. group size:  1688    Converged:          Yes            
Mean group size:  1409.6                                     
-------------------------------------------------------------
                  Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept          6.106    2.996   2.038 0.042  0.233 11.979
epoch2_indicator  -2.317    0.175 -13.240 0.000 -2.660 -1.974
Group Var         44.309                                     
animal_day Var     7.586    0.380                            
pair_id Var        0.287    0.123                            



/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)




distance_net_cm early_air_off_recovery
Early air-off recovery
            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: distance_net_cm
No. Observations: 7034    Method:             ML             
No. Groups:       5       Scale:              53.8904        
Min. group size:  872     Log-Likelihood:     -24119.4034    
Max. group size:  1678    Converged:          Yes            
Mean group size:  1406.8                                     
-------------------------------------------------------------
                  Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept          3.860    3.007   1.284 0.199 -2.034  9.754
epoch2_indicator  -2.136    0.175 -12.199 0.000 -2.479 -1.792
Group Var         44.779                                     
animal_day Var     5.342    0.181                            
pair_id Var        0.293    0.127                            



/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)




distance_net_cm late_air_off_recovery_to_next_baseline
Late air-off recovery / next pre-air baseline
            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: distance_net_cm
No. Observations: 6916    Method:             ML             
No. Groups:       5       Scale:              4.3440         
Min. group size:  850     Log-Likelihood:     -15524.3012    
Max. group size:  1658    Converged:          Yes            
Mean group size:  1383.2                                     
-------------------------------------------------------------
                  Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept          1.754    0.774   2.265 0.024  0.236  3.271
epoch2_indicator  -0.530    0.050 -10.573 0.000 -0.628 -0.432
Group Var          2.810                                     
animal_day Var     2.761    0.218                            
pair_id Var        0.699    0.047        

In [ ]:
def extract_air_basic_result(model, outcome, comparison, interpretation):
    fe = model.fe_params
    bse = model.bse_fe
    pvals = model.pvalues.loc[fe.index]
    conf = model.conf_int().loc[fe.index]

    term = "epoch2_indicator"

    return {
        "outcome": outcome,
        "comparison": comparison,
        "interpretation": interpretation,
        "n_obs": int(model.nobs),
        "n_animals": len(model.model.group_labels),
        "estimate_epoch2_minus_epoch1": fe.get(term, np.nan),
        "SE": bse.get(term, np.nan),
        "z": fe.get(term, np.nan) / bse.get(term, np.nan),
        "p": pvals.get(term, np.nan),
        "ci_low": conf.loc[term, 0] if term in conf.index else np.nan,
        "ci_high": conf.loc[term, 1] if term in conf.index else np.nan,
        "converged": model.converged,
    }


basic_rows = []

for outcome in air_outcomes:
    for comp in air_comparisons:
        key = (outcome, comp["comparison"])
        if key not in air_basic_models:
            continue

        basic_rows.append(
            extract_air_basic_result(
                air_basic_models[key],
                outcome=outcome,
                comparison=comp["comparison"],
                interpretation=comp["interpretation"],
            )
        )

air_basic_results_df = pd.DataFrame(basic_rows)

air_basic_results_df

In [ ]:
outcome_order = {
    "frac_moving": 1,
    "frac_forward": 2,
    "frac_stationary": 3,
    "mean_speed_path_cms": 4,
    "mean_speed_net_cms": 5,
    "distance_path_cm": 6,
    "distance_net_cm": 7,
    "dist_path_cm": 6,
    "dist_net_cm": 7,
}

comparison_order = {
    "air_onset_transition": 1,
    "early_air_on_progression": 2,
    "late_air_on_progression": 3,
    "air_offset_transition": 4,
    "early_air_off_recovery": 5,
    "late_air_off_recovery_to_next_baseline": 6,
}

air_basic_results_df["outcome_order"] = air_basic_results_df["outcome"].map(outcome_order)
air_basic_results_df["comparison_order"] = air_basic_results_df["comparison"].map(comparison_order)

air_basic_results_df = (
    air_basic_results_df
    .sort_values(["outcome_order", "comparison_order"])
    .reset_index(drop=True)
)

air_basic_results_df

Omnibus Model

In [12]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

# --------------------------------------------------
# Full air-cycle epoch order
# --------------------------------------------------

air_cycle_order = [
    "pre_air_on",
    "post_air_on",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_air_on_next",
]


def make_air_cycle_df(air_window_df):
    """
    Build long-format full air-cycle dataframe.

    One cycle = trial N.

    Includes:
        pre_air_on from trial N
        ...
        post_air_off_mid from trial N
        pre_air_on_next from trial N+1
    """

    df = air_window_df.copy()

    # current-trial epochs
    cur = df[
        df["epoch_label"].isin([
            "pre_air_on",
            "post_air_on",
            "pre_air_on_mid",
            "post_air_on_mid",
            "pre_air_off",
            "post_air_off",
            "pre_air_off_mid",
            "post_air_off_mid",
        ])
    ].copy()

    cur["cycle_trial"] = cur["event_number"].astype(int)
    cur["air_cycle_epoch"] = cur["epoch_label"]

    # next pre-air baseline:
    # pre_air_on from trial N+1 gets assigned to cycle_trial N
    nxt = df[df["epoch_label"] == "pre_air_on"].copy()
    nxt["cycle_trial"] = nxt["event_number"].astype(int) - 1
    nxt["air_cycle_epoch"] = "pre_air_on_next"

    # keep only valid next-baseline rows
    nxt = nxt[nxt["cycle_trial"] >= 1].copy()

    air_cycle_df = pd.concat([cur, nxt], ignore_index=True)

    # cycle/session IDs
    air_cycle_df["animal_day"] = (
        air_cycle_df["animal"].astype(str) + ":" +
        air_cycle_df["date"].astype(str)
    )

    air_cycle_df["cycle_id"] = (
        air_cycle_df["animal"].astype(str) + ":" +
        air_cycle_df["date"].astype(str) + ":" +
        air_cycle_df["cycle_trial"].astype(str)
    )

    # --------------------------------------------------
    # Use pre_air_on of the cycle as the cycle-level time/session reference
    # --------------------------------------------------
    cycle_info = (
        air_cycle_df[air_cycle_df["air_cycle_epoch"] == "pre_air_on"]
        [["animal", "date", "cycle_trial", "session_time_min", "exposure_session"]]
        .drop_duplicates()
        .rename(columns={
            "session_time_min": "cycle_session_time_min",
            "exposure_session": "cycle_exposure_session",
        })
    )

    air_cycle_df = air_cycle_df.merge(
        cycle_info,
        on=["animal", "date", "cycle_trial"],
        how="left"
    )

    # ordered categorical
    air_cycle_df["air_cycle_epoch"] = pd.Categorical(
        air_cycle_df["air_cycle_epoch"],
        categories=air_cycle_order,
        ordered=True,
    )

    # center predictors for later time-dependent model
    air_cycle_df["cycle_exposure_session_c"] = (
        air_cycle_df["cycle_exposure_session"] -
        air_cycle_df["cycle_exposure_session"].mean()
    )

    air_cycle_df["cycle_session_10m_c"] = (
        air_cycle_df["cycle_session_time_min"] -
        air_cycle_df["cycle_session_time_min"].mean()
    ) / 10.0

    return air_cycle_df.reset_index(drop=True)


air_cycle_df = make_air_cycle_df(air_window_df)

print(air_cycle_df.shape)
print(air_cycle_df["air_cycle_epoch"].value_counts().sort_index())

(31609, 70)
air_cycle_epoch
pre_air_on          3532
post_air_on         3535
pre_air_on_mid      3534
post_air_on_mid     3535
pre_air_off         3535
post_air_off        3525
pre_air_off_mid     3517
post_air_off_mid    3516
pre_air_on_next     3380
Name: count, dtype: int64


In [13]:
air_cycle_df[
    [
        "animal", "date", "cycle_trial",
        "air_cycle_epoch",
        "event_number",
        "cycle_exposure_session",
        "cycle_session_time_min",
    ]
].head(30)

,animal,date,cycle_trial,air_cycle_epoch,event_number,cycle_exposure_session,cycle_session_time_min
0,NML_04,2026_01_12,0,post_air_off_mid,0,NaN,NaN
1,NML_04,2026_01_12,0,pre_air_off_mid,0,NaN,NaN
2,NML_04,2026_01_12,0,post_air_off,0,NaN,NaN
3,NML_04,2026_01_12,0,pre_air_off,0,NaN,NaN
4,NML_04,2026_01_12,0,post_air_on_mid,0,NaN,NaN
5,NML_04,2026_01_12,0,pre_air_on_mid,0,NaN,NaN
6,NML_04,2026_01_12,0,post_air_on,0,NaN,NaN
7,NML_04,2026_01_12,1,post_air_off_mid,1,1.0,0.332607
8,NML_04,2026_01_12,1,pre_air_off_mid,1,1.0,0.332607
9,NML_04,2026_01_12,1,post_air_off,1,1.0,0.332607


In [39]:
def fit_air_cycle_omnibus_model(
    air_cycle_df,
    outcome,
    include_cycle_random=True,
):
    """
    Omnibus model across the full air cycle.

    outcome ~ C(air_cycle_epoch)

    Reference epoch should be pre_air_on because it is the first category.
    """

    df = air_cycle_df.copy()
    df = df[np.isfinite(df[outcome])].copy()

    if include_cycle_random:
        vc = {
            "animal_day": "0 + C(animal_day)",
            "cycle_id": "0 + C(cycle_id)",
        }
    else:
        vc = {
            "animal_day": "0 + C(animal_day)",
        }

    model = smf.mixedlm(
        f"{outcome} ~ C(air_cycle_epoch)",
        data=df,
        groups=df["animal"],
        re_formula="1",
        vc_formula=vc,
    ).fit(
        reml=False,
        method="lbfgs",
        maxiter=1000,
    )

    return model

In [40]:
main_air_outcomes = [
    "frac_moving",
    "frac_forward",
    "mean_speed_path_cms",
    "mean_speed_net_cms",
]

main_air_outcomes = [
    x for x in main_air_outcomes
    if x in air_cycle_df.columns
]

air_cycle_models = {}

for outcome in main_air_outcomes:
    print("\n\n==============================")
    print(outcome)
    print("==============================")

    try:
        model = fit_air_cycle_omnibus_model(
            air_cycle_df,
            outcome=outcome,
            include_cycle_random=True,
        )

        air_cycle_models[outcome] = model
        print(model.summary())

    except Exception as e:
        print(f"[ERROR] {outcome}: {e}")



frac_moving


/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


                      Mixed Linear Model Regression Results
Model:                    MixedLM         Dependent Variable:         frac_moving
No. Observations:         31609           Method:                     ML         
No. Groups:               5               Scale:                      0.0919     
Min. group size:          3918            Log-Likelihood:             -7889.2536 
Max. group size:          7562            Converged:                  Yes        
Mean group size:          6321.8                                                 
---------------------------------------------------------------------------------
                                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------
Intercept                               0.273    0.121  2.257 0.024  0.036  0.510
C(air_cycle_epoch)[T.post_air_on]       0.501    0.007 69.438 0.000  0.487  0.515
C(air_cycle_epoch)[T.pre_air_on_mid]  

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                      Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        frac_forward
No. Observations:          31609          Method:                    ML          
No. Groups:                5              Scale:                     0.1001      
Min. group size:           3918           Log-Likelihood:            -9014.6526  
Max. group size:           7562           Converged:                 Yes         
Mean group size:           6321.8                                                
---------------------------------------------------------------------------------
                                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------
Intercept                               0.252    0.024 10.625 0.000  0.206  0.299
C(air_cycle_epoch)[T.post_air_on]       0.403    0.008 53.614 0.000  0.389  0.418
C(air_cycle_epoch)[T.pre_air_on_mid]  

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


                       Mixed Linear Model Regression Results
Model:                   MixedLM       Dependent Variable:       mean_speed_net_cms
No. Observations:        31609         Method:                   ML                
No. Groups:              5             Scale:                    209.2914          
Min. group size:         3918          Log-Likelihood:           -129395.2857      
Max. group size:         7562          Converged:                Yes               
Mean group size:         6321.8                                                    
-----------------------------------------------------------------------------------
                                        Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
-----------------------------------------------------------------------------------
Intercept                                1.134    6.116  0.185 0.853 -10.854 13.122
C(air_cycle_epoch)[T.post_air_on]        1.270    0.344  3.689 0.000   0.595  1.944
C(air_cycle_epo

In [41]:
air_adjacent_contrasts = [
    ("pre_air_on", "post_air_on", "Air-onset transition"),
    ("post_air_on", "pre_air_on_mid", "Early air-on progression"),
    ("pre_air_on_mid", "post_air_on_mid", "Air-on midpoint transition"),
    ("post_air_on_mid", "pre_air_off", "Late air-on progression"),
    ("pre_air_off", "post_air_off", "Air-offset transition"),
    ("post_air_off", "pre_air_off_mid", "Early air-off recovery"),
    ("pre_air_off_mid", "post_air_off_mid", "Air-off midpoint transition"),
    ("post_air_off_mid", "pre_air_on_next", "Late air-off recovery / next baseline"),
]


def _epoch_term_name(result, epoch):
    """
    Return the fixed-effect term name for an air_cycle_epoch level.

    Baseline level has no term and returns None.
    """
    if epoch == "pre_air_on":
        return None

    target = f"C(air_cycle_epoch)[T.{epoch}]"

    if target in result.fe_params.index:
        return target

    # fallback search
    matches = [
        term for term in result.fe_params.index
        if f"T.{epoch}" in term
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(f"Could not find term for epoch: {epoch}")


def contrast_air_cycle_epochs(result, epoch1, epoch2, label="contrast"):
    """
    Contrast epoch2 - epoch1 from omnibus air-cycle model.
    """

    fe = result.fe_params
    cov = result.cov_params().loc[fe.index, fe.index]

    L = pd.Series(0.0, index=fe.index)

    term1 = _epoch_term_name(result, epoch1)
    term2 = _epoch_term_name(result, epoch2)

    # epoch2 - epoch1
    if term2 is not None:
        L[term2] += 1.0
    if term1 is not None:
        L[term1] -= 1.0

    estimate = float(np.dot(L.values, fe.values))
    se = float(np.sqrt(np.dot(L.values, np.dot(cov.values, L.values))))
    z = estimate / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    return {
        "contrast": label,
        "epoch1": epoch1,
        "epoch2": epoch2,
        "estimate_epoch2_minus_epoch1": estimate,
        "SE": se,
        "z": z,
        "p": p,
        "ci_low": estimate - 1.96 * se,
        "ci_high": estimate + 1.96 * se,
    }


def extract_air_cycle_adjacent_contrasts(model, outcome):
    rows = []

    for epoch1, epoch2, label in air_adjacent_contrasts:
        row = contrast_air_cycle_epochs(
            model,
            epoch1=epoch1,
            epoch2=epoch2,
            label=label,
        )
        row["outcome"] = outcome
        rows.append(row)

    return pd.DataFrame(rows)

In [42]:
air_cycle_contrast_tables = []

for outcome, model in air_cycle_models.items():
    tmp = extract_air_cycle_adjacent_contrasts(model, outcome)
    air_cycle_contrast_tables.append(tmp)

air_cycle_contrasts_df = pd.concat(
    air_cycle_contrast_tables,
    ignore_index=True
)

air_cycle_contrasts_df

,contrast,epoch1,epoch2,estimate_epoch2_minus_epoch1,SE,z,p,ci_low,ci_high,outcome
0,Air-onset transition,pre_air_on,post_air_on,0.500773,0.007212,69.437825,0.000000e+00,0.486638,0.514908,frac_moving
1,Early air-on progression,post_air_on,pre_air_on_mid,0.048449,0.007211,6.718941,1.830491e-11,0.034316,0.062582,frac_moving
2,Air-on midpoint transition,pre_air_on_mid,post_air_on_mid,0.045448,0.007211,6.302772,2.923688e-10,0.031315,0.059581,frac_moving
3,Late air-on progression,post_air_on_mid,pre_air_off,0.094007,0.007210,13.038056,0.000000e+00,0.079875,0.108140,frac_moving
4,Air-offset transition,pre_air_off,post_air_off,-0.044892,0.007216,-6.221477,4.924976e-10,-0.059034,-0.030749,frac_moving
5,Early air-off recovery,post_air_off,pre_air_off_mid,-0.515500,0.007225,-71.352740,0.000000e+00,-0.529660,-0.501340,frac_moving
6,Air-off midpoint transition,pre_air_off_mid,post_air_off_mid,-0.026158,0.007229,-3.618415,2.964125e-04,-0.040327,-0.011989,frac_moving
7,Late air-off recovery / next baseline,post_air_off_mid,pre_air_on_next,-0.106972,0.007304,-14.644814,0.000000e+00,-0.121289,-0.092655,frac_moving
8,Air-onset transition,pre_air_on,post_air_on,0.403463,0.007525,53.613893,0.000000e+00,0.388714,0.418213,frac_forward
9,Early air-on progression,post_air_on,pre_air_on_mid,0.121013,0.007524,16.083079,0.000000e+00,0.106266,0.135761,frac_forward


In [43]:
def fit_air_cycle_time_model(
    air_cycle_df,
    outcome,
    include_cycle_random=True,
):
    """
    Time-dependent omnibus model.

    Tests whether air-cycle structure changes with:
        cycle_exposure_session_c
        cycle_session_10m_c
    """

    df = air_cycle_df.copy()

    df = df[
        np.isfinite(df[outcome]) &
        np.isfinite(df["cycle_exposure_session_c"]) &
        np.isfinite(df["cycle_session_10m_c"])
    ].copy()

    if include_cycle_random:
        vc = {
            "animal_day": "0 + C(animal_day)",
            "cycle_id": "0 + C(cycle_id)",
        }
    else:
        vc = {
            "animal_day": "0 + C(animal_day)",
        }

    formula = (
        f"{outcome} ~ C(air_cycle_epoch) * "
        f"cycle_exposure_session_c * cycle_session_10m_c"
    )

    model = smf.mixedlm(
        formula,
        data=df,
        groups=df["animal"],
        re_formula="1",
        vc_formula=vc,
    ).fit(
        reml=False,
        method="lbfgs",
        maxiter=1000,
    )

    return model

In [44]:
m_cycle_time_frac_moving = fit_air_cycle_time_model(
    air_cycle_df,
    outcome="frac_moving",
    include_cycle_random=True,
)

print(m_cycle_time_frac_moving.summary())

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                                            Mixed Linear Model Regression Results
Model:                                   MixedLM                        Dependent Variable:                        frac_moving
No. Observations:                        31584                          Method:                                    ML         
No. Groups:                              5                              Scale:                                     0.0887     
Min. group size:                         3918                           Log-Likelihood:                            -7296.1970 
Max. group size:                         7557                           Converged:                                 Yes        
Mean group size:                         6316.8                                                                               
------------------------------------------------------------------------------------------------------------------------------
                             

In [45]:
air_cycle_time_models = {}

for outcome in main_air_outcomes:
    print("\n\n==============================")
    print(outcome)
    print("==============================")

    try:
        model = fit_air_cycle_time_model(
            air_cycle_df,
            outcome=outcome,
            include_cycle_random=True,
        )

        air_cycle_time_models[outcome] = model
        print(model.summary())

    except Exception as e:
        print(f"[ERROR] {outcome}: {e}")



frac_moving


/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                                            Mixed Linear Model Regression Results
Model:                                   MixedLM                        Dependent Variable:                        frac_moving
No. Observations:                        31584                          Method:                                    ML         
No. Groups:                              5                              Scale:                                     0.0887     
Min. group size:                         3918                           Log-Likelihood:                            -7296.1970 
Max. group size:                         7557                           Converged:                                 Yes        
Mean group size:                         6316.8                                                                               
------------------------------------------------------------------------------------------------------------------------------
                             

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                                            Mixed Linear Model Regression Results
Model:                                    MixedLM                       Dependent Variable:                       frac_forward
No. Observations:                         31584                         Method:                                   ML          
No. Groups:                               5                             Scale:                                    0.0956      
Min. group size:                          3918                          Log-Likelihood:                           -8365.5809  
Max. group size:                          7557                          Converged:                                Yes         
Mean group size:                          6316.8                                                                              
------------------------------------------------------------------------------------------------------------------------------
                             

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


                                             Mixed Linear Model Regression Results
Model:                                 MixedLM                      Dependent Variable:                      mean_speed_path_cms
No. Observations:                      31584                        Method:                                  ML                 
No. Groups:                            5                            Scale:                                   206.6515           
Min. group size:                       3918                         Log-Likelihood:                          -129080.2321       
Max. group size:                       7557                         Converged:                               Yes                
Mean group size:                       6316.8                                                                                   
--------------------------------------------------------------------------------------------------------------------------------
              

/home/nmldata2/miniconda3/envs/air_wheel/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2262: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


                                             Mixed Linear Model Regression Results
Model:                                  MixedLM                      Dependent Variable:                      mean_speed_net_cms
No. Observations:                       31584                        Method:                                  ML                
No. Groups:                             5                            Scale:                                   208.5802          
Min. group size:                        3918                         Log-Likelihood:                          -129229.2275      
Max. group size:                        7557                         Converged:                               Yes               
Mean group size:                        6316.8                                                                                  
--------------------------------------------------------------------------------------------------------------------------------
              

In [ ]:
from pathlib import Path

air_results_dir = Path(pdata_root) / "results"
air_results_dir.mkdir(parents=True, exist_ok=True)

air_cycle_contrasts_df.to_csv(
    air_results_dir / "air_cycle_omnibus_adjacent_contrasts.csv",
    index=False,
)

In [1]:
%load_ext rpy2.ipython

In [14]:
%R -i air_cycle_df

In [15]:
%%R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

air_cycle_df$animal <- factor(air_cycle_df$animal)
air_cycle_df$animal_day <- factor(air_cycle_df$animal_day)
air_cycle_df$cycle_id <- factor(air_cycle_df$cycle_id)

air_cycle_df$air_cycle_epoch <- factor(
  air_cycle_df$air_cycle_epoch,
  levels = c(
    "pre_air_on",
    "post_air_on",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_air_on_next"
  )
)

R[write to console]: Loading required package: Matrix

R[write to console]: 
Attaching package: ‘lmerTest’


R[write to console]: The following object is masked from ‘package:lme4’:

    lmer


R[write to console]: The following object is masked from ‘package:stats’:

    step


R[write to console]: Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [19]:
%%R

air_cycle_df$air_cycle_epoch <- factor(
  as.character(air_cycle_df$air_cycle_epoch),
  levels = c(
    "pre_air_on",
    "post_air_on",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_air_on_next"
  ),
  ordered = FALSE
)

In [20]:
%%R

m_frac_moving <- lmer(
  frac_moving ~ air_cycle_epoch * cycle_exposure_session_c * cycle_session_10m_c +
    (1 | animal) +
    (1 | animal_day) +
    (1 | cycle_id),
  data = air_cycle_df,
  REML = FALSE
)

summary(m_frac_moving)

Linear mixed model fit by maximum likelihood . t-tests use Satterthwaite's
  method [lmerModLmerTest]
Formula: 
frac_moving ~ air_cycle_epoch * cycle_exposure_session_c * cycle_session_10m_c +  
    (1 | animal) + (1 | animal_day) + (1 | cycle_id)
   Data: air_cycle_df

      AIC       BIC    logLik -2*log(L)  df.resid 
  14672.4   15006.8   -7296.2   14592.4     31544 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4176 -0.6753  0.0390  0.5104  3.2582 

Random effects:
 Groups     Name        Variance Std.Dev.
 cycle_id   (Intercept) 0.004092 0.06397 
 animal_day (Intercept) 0.006146 0.07839 
 animal     (Intercept) 0.001819 0.04264 
 Residual               0.088748 0.29791 
Number of obs: 31584, groups:  cycle_id, 3532; animal_day, 78; animal, 5

Fixed effects:
                                                                               Estimate
(Intercept)                                                                   2.807e-01
air_cycle_epochpost_air_on       

R[write to console]: 
Correlation matrix not shown by default, as p = 36 > 12.
Use print(object, correlation=TRUE)  or
    vcov(object)        if you need it




In [23]:
%%R
emm_frac_moving <- emmeans(
  m_frac_moving,
  ~ air_cycle_epoch,
  at = list(
    cycle_exposure_session_c = 0,
    cycle_session_10m_c = 0
  )
)

R[write to console]: Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 31584' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 31584)' or larger];
but be warned that this may result in large computation time and memory use.

R[write to console]: Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 31584' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 31584)' or larger];
but be warned that this may result in large computation time and memory use.

R[write to console]: NOTE: Results may be misleading due to involvement in interactions



In [24]:
%%R

emm_frac_moving <- emmeans(m_frac_moving, ~ air_cycle_epoch)

contrast_list <- list(
  air_onset_transition = c(-1, 1, 0, 0, 0, 0, 0, 0, 0),
  early_air_on_progression = c(0, -1, 1, 0, 0, 0, 0, 0, 0),
  air_on_mid_transition = c(0, 0, -1, 1, 0, 0, 0, 0, 0),
  late_air_on_progression = c(0, 0, 0, -1, 1, 0, 0, 0, 0),
  air_offset_transition = c(0, 0, 0, 0, -1, 1, 0, 0, 0),
  early_air_off_recovery = c(0, 0, 0, 0, 0, -1, 1, 0, 0),
  air_off_mid_transition = c(0, 0, 0, 0, 0, 0, -1, 1, 0),
  late_air_off_recovery_to_next_baseline = c(0, 0, 0, 0, 0, 0, 0, -1, 1)
)

contrast(emm_frac_moving, contrast_list)

R[write to console]: Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 31584' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 31584)' or larger];
but be warned that this may result in large computation time and memory use.

R[write to console]: Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 31584' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 31584)' or larger];
but be warned that this may result in large computation time and memory use.

R[write to console]: NOTE: Results may be misleading due to involvement in interactions



 contrast                               estimate      SE  df z.ratio p.value
 air_onset_transition                     0.4995 0.00713 Inf  70.099  <.0001
 early_air_on_progression                 0.0479 0.00713 Inf   6.726  <.0001
 air_on_mid_transition                    0.0455 0.00713 Inf   6.387  <.0001
 late_air_on_progression                  0.0943 0.00713 Inf  13.233  <.0001
 air_offset_transition                   -0.0452 0.00713 Inf  -6.343  <.0001
 early_air_off_recovery                  -0.5143 0.00714 Inf -72.006  <.0001
 air_off_mid_transition                  -0.0258 0.00715 Inf  -3.614  0.0003
 late_air_off_recovery_to_next_baseline  -0.1065 0.00722 Inf -14.754  <.0001

Degrees-of-freedom method: asymptotic 


In [25]:
%%R

r2(m_frac_moving)

# R2 for Mixed Models

  Conditional R2: 0.511
     Marginal R2: 0.445


In [27]:
%%R -o air_cycle_contrasts_R -o air_cycle_r2_R -o air_cycle_fixed_R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

# Use asymptotic df for large models
emm_options(lmer.df = "asymptotic")

# --------------------------------------------------
# Make sure grouping variables are factors
# --------------------------------------------------

air_cycle_df$animal <- factor(air_cycle_df$animal)
air_cycle_df$animal_day <- factor(air_cycle_df$animal_day)
air_cycle_df$cycle_id <- factor(air_cycle_df$cycle_id)

# Important: unordered factor, with pre_air_on as reference
air_cycle_df$air_cycle_epoch <- factor(
  as.character(air_cycle_df$air_cycle_epoch),
  levels = c(
    "pre_air_on",
    "post_air_on",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_air_on_next"
  ),
  ordered = FALSE
)

# --------------------------------------------------
# Outcomes to analyze
# --------------------------------------------------

air_outcomes_R <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

# Keep only outcomes that actually exist
air_outcomes_R <- air_outcomes_R[air_outcomes_R %in% names(air_cycle_df)]

print(air_outcomes_R)

# --------------------------------------------------
# Planned adjacent contrasts
# --------------------------------------------------

contrast_list <- list(
  air_onset_transition = c(-1, 1, 0, 0, 0, 0, 0, 0, 0),
  early_air_on_progression = c(0, -1, 1, 0, 0, 0, 0, 0, 0),
  air_on_mid_transition = c(0, 0, -1, 1, 0, 0, 0, 0, 0),
  late_air_on_progression = c(0, 0, 0, -1, 1, 0, 0, 0, 0),
  air_offset_transition = c(0, 0, 0, 0, -1, 1, 0, 0, 0),
  early_air_off_recovery = c(0, 0, 0, 0, 0, -1, 1, 0, 0),
  air_off_mid_transition = c(0, 0, 0, 0, 0, 0, -1, 1, 0),
  late_air_off_recovery_to_next_baseline = c(0, 0, 0, 0, 0, 0, 0, -1, 1)
)

# --------------------------------------------------
# Function to fit one outcome model
# --------------------------------------------------

fit_air_cycle_outcome <- function(outcome_name) {

  cat("\n\n==============================\n")
  cat(outcome_name, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome_name,
    " ~ air_cycle_epoch * cycle_exposure_session_c * cycle_session_10m_c + ",
    "(1 | animal) + (1 | animal_day) + (1 | cycle_id)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = air_cycle_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  print(summary(m))

  # Estimated marginal means at mean exposure session and mean session time
  emm <- emmeans(
    m,
    ~ air_cycle_epoch,
    at = list(
      cycle_exposure_session_c = 0,
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  # Planned adjacent contrasts
  con <- as.data.frame(
    contrast(
      emm,
      contrast_list,
      adjust = "none"
    )
  )

  con$outcome <- outcome_name

  # R2
  r2tmp <- as.data.frame(performance::r2(m))
  r2tmp$outcome <- outcome_name

  # Fixed effects table
  fixed <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )
  fixed$outcome <- outcome_name

  return(
    list(
      model = m,
      contrasts = con,
      r2 = r2tmp,
      fixed = fixed
    )
  )
}

# --------------------------------------------------
# Run all outcomes
# --------------------------------------------------

air_models_R <- list()
contrast_list_out <- list()
r2_list_out <- list()
fixed_list_out <- list()

for (outcome_name in air_outcomes_R) {

  result <- fit_air_cycle_outcome(outcome_name)

  air_models_R[[outcome_name]] <- result$model
  contrast_list_out[[outcome_name]] <- result$contrasts
  r2_list_out[[outcome_name]] <- result$r2
  fixed_list_out[[outcome_name]] <- result$fixed
}

air_cycle_contrasts_R <- bind_rows(contrast_list_out)
air_cycle_r2_R <- bind_rows(r2_list_out)
air_cycle_fixed_R <- bind_rows(fixed_list_out)

air_cycle_contrasts_R <- air_cycle_contrasts_R %>%
  select(outcome, contrast, estimate, SE, df, z.ratio, p.value)

air_cycle_r2_R <- air_cycle_r2_R %>%
  relocate(outcome)

air_cycle_fixed_R <- air_cycle_fixed_R %>%
  relocate(outcome)

print(air_cycle_contrasts_R)
print(air_cycle_r2_R)

[1] "frac_moving"         "frac_forward"        "mean_speed_path_cms"
[4] "mean_speed_net_cms"  "distance_path_cm"    "distance_net_cm"    


frac_moving 
Linear mixed model fit by maximum likelihood . t-tests use Satterthwaite's
  method [lmerModLmerTest]
Formula: as.formula(formula_text)
   Data: air_cycle_df
Control: lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  14672.4   15006.8   -7296.2   14592.4     31544 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4176 -0.6753  0.0390  0.5104  3.2582 

Random effects:
 Groups     Name        Variance Std.Dev.
 cycle_id   (Intercept) 0.004092 0.06397 
 animal_day (Intercept) 0.006146 0.07839 
 animal     (Intercept) 0.001819 0.04265 
 Residual               0.088748 0.29791 
Number of obs: 31584, groups:  cycle_id, 3532; animal_day, 78; animal, 5

Fixed effects:
                                                                               Estimate
(In

R[write to console]: 
Correlation matrix not shown by default, as p = 36 > 12.
Use print(summary(m), correlation=TRUE)  or
    vcov(summary(m))        if you need it


R[write to console]: NOTE: Results may be misleading due to involvement in interactions





frac_forward 
Linear mixed model fit by maximum likelihood . t-tests use Satterthwaite's
  method [lmerModLmerTest]
Formula: as.formula(formula_text)
   Data: air_cycle_df
Control: lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  16811.2   17145.6   -8365.6   16731.2     31544 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.2639 -0.7028  0.0238  0.5834  3.2957 

Random effects:
 Groups     Name        Variance Std.Dev.
 cycle_id   (Intercept) 0.003468 0.05889 
 animal_day (Intercept) 0.006467 0.08042 
 animal     (Intercept) 0.002991 0.05469 
 Residual               0.095637 0.30925 
Number of obs: 31584, groups:  cycle_id, 3532; animal_day, 78; animal, 5

Fixed effects:
                                                                               Estimate
(Intercept)                                                                   2.576e-01
air_cycle_epochpost_air_on                            

R[write to console]: 
Correlation matrix not shown by default, as p = 36 > 12.
Use print(summary(m), correlation=TRUE)  or
    vcov(summary(m))        if you need it


R[write to console]: NOTE: Results may be misleading due to involvement in interactions





mean_speed_path_cms 
Linear mixed model fit by maximum likelihood . t-tests use Satterthwaite's
  method [lmerModLmerTest]
Formula: as.formula(formula_text)
   Data: air_cycle_df
Control: lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
 258215.7  258550.1 -129067.9  258135.7     31544 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
 -1.360  -0.107  -0.031   0.088 169.383 

Random effects:
 Groups     Name        Variance Std.Dev.
 cycle_id   (Intercept)   0.2219  0.4711 
 animal_day (Intercept)   1.5455  1.2432 
 animal     (Intercept)   0.3456  0.5879 
 Residual               206.5536 14.3720 
Number of obs: 31584, groups:  cycle_id, 3532; animal_day, 78; animal, 5

Fixed effects:
                                                                               Estimate
(Intercept)                                                                   1.370e+00
air_cycle_epochpost_air_on                     

...



distance_path_cm 
Linear mixed model fit by maximum likelihood . t-tests use Satterthwaite's
  method [lmerModLmerTest]
Formula: as.formula(formula_text)
   Data: air_cycle_df
Control: lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
 258219.2  258553.7 -129069.6  258139.2     31544 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
 -1.361  -0.107  -0.031   0.088 169.374 

Random effects:
 Groups     Name        Variance Std.Dev.
 cycle_id   (Intercept)   0.2209  0.4700 
 animal_day (Intercept)   1.5460  1.2434 
 animal     (Intercept)   0.3441  0.5866 
 Residual               206.5776 14.3728 
Number of obs: 31584, groups:  cycle_id, 3532; animal_day, 78; animal, 5

Fixed effects:
                                                                               Estimate
(Intercept)                                                                   1.374e+00
air_cycle_epochpost_air_on                        

R[write to console]: 
Correlation matrix not shown by default, as p = 36 > 12.
Use print(summary(m), correlation=TRUE)  or
    vcov(summary(m))        if you need it


R[write to console]: NOTE: Results may be misleading due to involvement in interactions





distance_net_cm 
Linear mixed model fit by maximum likelihood . t-tests use Satterthwaite's
  method [lmerModLmerTest]
Formula: as.formula(formula_text)
   Data: air_cycle_df
Control: lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
 258516.3  258850.7 -129218.2  258436.3     31544 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-168.990   -0.108   -0.026    0.109    1.480 

Random effects:
 Groups     Name        Variance Std.Dev.
 cycle_id   (Intercept)   0.8122  0.9012 
 animal_day (Intercept)   1.8341  1.3543 
 animal     (Intercept)   0.7018  0.8377 
 Residual               207.8850 14.4182 
Number of obs: 31584, groups:  cycle_id, 3532; animal_day, 78; animal, 5

Fixed effects:
                                                                               Estimate
(Intercept)                                                                   1.216e+00
air_cycle_epochpost_air_on               

R[write to console]: 
Correlation matrix not shown by default, as p = 36 > 12.
Use print(summary(m), correlation=TRUE)  or
    vcov(summary(m))        if you need it


R[write to console]: NOTE: Results may be misleading due to involvement in interactions



               outcome                               contrast    estimate
1          frac_moving                   air_onset_transition  0.49954039
2          frac_moving               early_air_on_progression  0.04793399
3          frac_moving                  air_on_mid_transition  0.04551561
4          frac_moving                late_air_on_progression  0.09430300
5          frac_moving                  air_offset_transition -0.04523749
6          frac_moving                 early_air_off_recovery -0.51430726
7          frac_moving                 air_off_mid_transition -0.02583070
8          frac_moving late_air_off_recovery_to_next_baseline -0.10653793
9         frac_forward                   air_onset_transition  0.40270597
10        frac_forward               early_air_on_progression  0.12001430
11        frac_forward                  air_on_mid_transition  0.06053917
12        frac_forward                late_air_on_progression  0.10820582
13        frac_forward                

In [28]:
%%R -o air_cycle_emmeans_R -o air_cycle_contrasts_R -o air_cycle_r2_R -o air_cycle_fixed_R -o air_cycle_session_trends_R -o air_cycle_session_time_trends_R -o air_cycle_model_info_R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

# --------------------------------------------------
# Use asymptotic df for emmeans because models are large
# --------------------------------------------------
emm_options(lmer.df = "asymptotic")

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

air_cycle_df$animal <- factor(air_cycle_df$animal)
air_cycle_df$animal_day <- factor(air_cycle_df$animal_day)
air_cycle_df$cycle_id <- factor(air_cycle_df$cycle_id)

air_cycle_df$air_cycle_epoch <- factor(
  as.character(air_cycle_df$air_cycle_epoch),
  levels = c(
    "pre_air_on",
    "post_air_on",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_air_on_next"
  ),
  ordered = FALSE
)

# --------------------------------------------------
# Outcomes
# --------------------------------------------------

air_outcomes_R <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

air_outcomes_R <- air_outcomes_R[air_outcomes_R %in% names(air_cycle_df)]

print(air_outcomes_R)

# --------------------------------------------------
# Planned adjacent contrasts across the air cycle
# --------------------------------------------------

contrast_list <- list(
  air_onset_transition = c(-1, 1, 0, 0, 0, 0, 0, 0, 0),
  early_air_on_progression = c(0, -1, 1, 0, 0, 0, 0, 0, 0),
  air_on_mid_transition = c(0, 0, -1, 1, 0, 0, 0, 0, 0),
  late_air_on_progression = c(0, 0, 0, -1, 1, 0, 0, 0, 0),
  air_offset_transition = c(0, 0, 0, 0, -1, 1, 0, 0, 0),
  early_air_off_recovery = c(0, 0, 0, 0, 0, -1, 1, 0, 0),
  air_off_mid_transition = c(0, 0, 0, 0, 0, 0, -1, 1, 0),
  late_air_off_recovery_to_next_baseline = c(0, 0, 0, 0, 0, 0, 0, -1, 1)
)

# --------------------------------------------------
# Function: fit model and extract all clean outputs
# --------------------------------------------------

fit_air_cycle_outcome <- function(outcome_name) {

  cat("\n\n==============================\n")
  cat("Fitting outcome:", outcome_name, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome_name,
    " ~ air_cycle_epoch * cycle_exposure_session_c * cycle_session_10m_c + ",
    "(1 | animal) + (1 | animal_day) + (1 | cycle_id)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = air_cycle_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  # --------------------------------------------------
  # Model info
  # --------------------------------------------------

  model_info <- data.frame(
    outcome = outcome_name,
    n_obs = nobs(m),
    n_animals = nlevels(air_cycle_df$animal),
    n_animal_day = nlevels(air_cycle_df$animal_day),
    n_cycle_id = nlevels(air_cycle_df$cycle_id),
    AIC = AIC(m),
    BIC = BIC(m),
    logLik = as.numeric(logLik(m)),
    singular = isSingular(m),
    stringsAsFactors = FALSE
  )

  # --------------------------------------------------
  # EMMs at mean exposure session and mean session time
  # --------------------------------------------------

  emm <- emmeans(
    m,
    ~ air_cycle_epoch,
    at = list(
      cycle_exposure_session_c = 0,
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  emm_df <- as.data.frame(emm)
  emm_df$outcome <- outcome_name

  # --------------------------------------------------
  # Planned adjacent contrasts at mean exposure/session time
  # --------------------------------------------------

  contrast_df <- as.data.frame(
    contrast(
      emm,
      contrast_list,
      adjust = "none"
    )
  )

  contrast_df$outcome <- outcome_name

  # --------------------------------------------------
  # R2
  # --------------------------------------------------

  r2_obj <- performance::r2_nakagawa(m)

  r2_df <- data.frame(
    outcome = outcome_name,
    R2_marginal = r2_obj$R2_marginal,
    R2_conditional = r2_obj$R2_conditional,
    stringsAsFactors = FALSE
  )

  # --------------------------------------------------
  # Fixed effects
  # --------------------------------------------------

  fixed_df <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )

  fixed_df$outcome <- outcome_name

  # --------------------------------------------------
  # Session/exposure-session trends by air-cycle epoch
  # Evaluated at mean within-session time
  # --------------------------------------------------

  session_trends <- emtrends(
    m,
    ~ air_cycle_epoch,
    var = "cycle_exposure_session_c",
    at = list(
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_trends_df <- as.data.frame(session_trends)
  session_trends_df$outcome <- outcome_name

  # --------------------------------------------------
  # Within-session-time trends by air-cycle epoch
  # Evaluated at mean exposure session
  # --------------------------------------------------

  session_time_trends <- emtrends(
    m,
    ~ air_cycle_epoch,
    var = "cycle_session_10m_c",
    at = list(
      cycle_exposure_session_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_time_trends_df <- as.data.frame(session_time_trends)
  session_time_trends_df$outcome <- outcome_name

  return(
    list(
      model = m,
      model_info = model_info,
      emmeans = emm_df,
      contrasts = contrast_df,
      r2 = r2_df,
      fixed = fixed_df,
      session_trends = session_trends_df,
      session_time_trends = session_time_trends_df
    )
  )
}

# --------------------------------------------------
# Run all models
# --------------------------------------------------

air_models_R <- list()
model_info_list <- list()
emmeans_list <- list()
contrasts_list <- list()
r2_list <- list()
fixed_list <- list()
session_trends_list <- list()
session_time_trends_list <- list()

for (outcome_name in air_outcomes_R) {

  result <- fit_air_cycle_outcome(outcome_name)

  air_models_R[[outcome_name]] <- result$model
  model_info_list[[outcome_name]] <- result$model_info
  emmeans_list[[outcome_name]] <- result$emmeans
  contrasts_list[[outcome_name]] <- result$contrasts
  r2_list[[outcome_name]] <- result$r2
  fixed_list[[outcome_name]] <- result$fixed
  session_trends_list[[outcome_name]] <- result$session_trends
  session_time_trends_list[[outcome_name]] <- result$session_time_trends
}

# --------------------------------------------------
# Combine output tables
# --------------------------------------------------

air_cycle_model_info_R <- bind_rows(model_info_list)
air_cycle_emmeans_R <- bind_rows(emmeans_list)
air_cycle_contrasts_R <- bind_rows(contrasts_list)
air_cycle_r2_R <- bind_rows(r2_list)
air_cycle_fixed_R <- bind_rows(fixed_list)
air_cycle_session_trends_R <- bind_rows(session_trends_list)
air_cycle_session_time_trends_R <- bind_rows(session_time_trends_list)

# --------------------------------------------------
# Reorder columns
# --------------------------------------------------

air_cycle_emmeans_R <- air_cycle_emmeans_R %>%
  relocate(outcome)

air_cycle_contrasts_R <- air_cycle_contrasts_R %>%
  relocate(outcome) %>%
  select(outcome, contrast, estimate, SE, df, z.ratio, p.value)

air_cycle_fixed_R <- air_cycle_fixed_R %>%
  relocate(outcome)

air_cycle_session_trends_R <- air_cycle_session_trends_R %>%
  relocate(outcome)

air_cycle_session_time_trends_R <- air_cycle_session_time_trends_R %>%
  relocate(outcome)

# --------------------------------------------------
# Print clean summaries
# --------------------------------------------------

cat("\n\n=== Model info ===\n")
print(air_cycle_model_info_R)

cat("\n\n=== Adjacent contrasts ===\n")
print(air_cycle_contrasts_R)

cat("\n\n=== R2 ===\n")
print(air_cycle_r2_R)

cat("\n\n=== Exposure-session trends by epoch ===\n")
print(air_cycle_session_trends_R)

cat("\n\n=== Within-session-time trends by epoch ===\n")
print(air_cycle_session_time_trends_R)

[1] "frac_moving"         "frac_forward"        "mean_speed_path_cms"
[4] "mean_speed_net_cms"  "distance_path_cm"    "distance_net_cm"    


Fitting outcome: frac_moving 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: frac_forward 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: mean_speed_path_cms 


...



Fitting outcome: distance_path_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: distance_net_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





=== Model info ===
              outcome n_obs n_animals n_animal_day n_cycle_id       AIC
1         frac_moving 31584         5           78       3536  14672.39
2        frac_forward 31584         5           78       3536  16811.16
3 mean_speed_path_cms 31584         5           78       3536 258215.73
4  mean_speed_net_cms 31584         5           78       3536 258515.36
5    distance_path_cm 31584         5           78       3536 258219.24
6     distance_net_cm 31584         5           78       3536 258516.32
        BIC      logLik singular
1  15006.81   -7296.197    FALSE
2  17145.58   -8365.581    FALSE
3 258550.15 -129067.866    FALSE
4 258849.78 -129217.682    FALSE
5 258553.66 -129069.622    FALSE
6 258850.74 -129218.159    FALSE


=== Adjacent contrasts ===
               outcome                               contrast    estimate
1          frac_moving                   air_onset_transition  0.49954039
2          frac_moving               early_air_on_progression  0.04